# 📈 Carteira Inteligente v5 — Previsão + Decisão + Projeção

**Melhorias v5:**
- Modelos separados por horizonte (D+1, D+2, D+3 independentes)
- Contexto de mercado: VIX + SPY como features
- Pesos do ensemble com decay temporal por horizonte
- Pasta `AnaliseV5 / AnaliseGraficos`

> ⚠️ Executa às **15h30 (Barcelona)** para máxima cobertura de dados.
> Mercados europeus e cripto já fecharam. US stocks validados na execução seguinte.

| Bloco | Descrição |
|-------|-----------|
| BLOCO 1 | Instalar dependências (só 1ª vez) |
| BLOCO 2 | Imports + seed |
| BLOCO 3 | Configuração da carteira |
| BLOCO 4 | Google Drive + CSV + pastas |
| BLOCO 5 | Download preços + câmbio + contexto (VIX, SPY) |
| BLOCO 6 | Feature Engineering |
| BLOCO 7 | Modelos ML D+1 / D+2 / D+3 independentes |
| BLOCO 8 | Validar previsões antigas + guardar novas |
| BLOCO 9 | Análise da carteira: G/P + breakeven |
| BLOCO 10 | Sinais de saída |
| BLOCO 11 | Projeção 1 / 3 / 5 / 10 anos |
| BLOCO 12 | Simulação DCA |
| BLOCO 13 | Visualizações → AnaliseGraficos |
| BLOCO 14 | Resumo final das previsões ML |

## BLOCO 1 — Instalações
Executa apenas na primeira vez num ambiente novo.

In [ ]:
# Descomenta na primeira execução num ambiente novo:
# !pip install yfinance scikit-learn matplotlib pandas numpy openpyxl

## BLOCO 2 — Imports e configuração global

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from datetime import datetime
from pathlib import Path

import yfinance as yf
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score

np.random.seed(42)

print("✅ Imports OK")
print(f"   Data de execução: {datetime.now().strftime('%d/%m/%Y %H:%M')}")

## BLOCO 3 — Configuração da Carteira
Edita aqui sempre que comprares ou venderes algo. O resto do notebook lê automaticamente daqui.

| Campo | Descrição |
|-------|-----------|
| `ticker` | Símbolo no Yahoo Finance |
| `unidades` | Quantas unidades tens |
| `preco_abertura` | Preço médio de compra |
| `moeda` | Moeda do ativo (USD / EUR / GBP) |
| `fee_euro` | Custo total abertura + fecho (eToro = 2€) |
| `aporte_mensal` | Quanto investes por mês (0 se não investes) |

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CONFIGURAÇÃO PRINCIPAL — Edita aqui os teus ativos
# ══════════════════════════════════════════════════════════════

ETORO = [
    {
        "ticker":        "LLY",
        "nome":          "Eli Lilly & Co",
        "unidades":      0.13048,
        "preco_abertura":1088.32,
        "moeda":         "USD",
        "fee_euro":      2.0,
        "aporte_mensal": 0,
    },
    {
        "ticker":        "NVDA",
        "nome":          "NVIDIA Corporation",
        "unidades":      0.48115,
        "preco_abertura":207.83523,
        "moeda":         "USD",
        "fee_euro":      2.0,
        "aporte_mensal": 0,
    },
    {
        "ticker":        "ALV.DE",
        "nome":          "Allianz SE",
        "unidades":      0.141595,
        "preco_abertura":354.30,
        "moeda":         "EUR",
        "fee_euro":      2.0,
        "aporte_mensal": 0,
    },
    {
        "ticker":        "BTC-USD",
        "nome":          "Bitcoin",
        "unidades":      0.000468,
        "preco_abertura":121992.01,
        "moeda":         "USD",
        "fee_euro":      0.0,   # cripto: fee via spread, não fixo
        "aporte_mensal": 0,
    },
    {
        "ticker":        "BABA",
        "nome":          "Alibaba ADR",
        "unidades":      0.1388,
        "preco_abertura":180.11,
        "moeda":         "USD",
        "fee_euro":      2.0,
        "aporte_mensal": 0,
    },
]

ETF_ACUMULACAO = [
    {
        "ticker":          "EXUS.L",
        "nome":            "MSCI World ex USA ETF",
        "unidades":        9.3331233141,
        "euros_investidos":350.00,
        "moeda":           "GBP",
        "gbp_pence":       False,   # preço em £ (libras)
        "fee_euro":        0.0,
        "aporte_mensal":   50,
    },
    {
        "ticker":          "ICGA.DE",
        "nome":            "MSCI China ETF",
        "unidades":        67.9584761758,
        "euros_investidos":349.99,
        "moeda":           "EUR",
        "gbp_pence":       False,
        "fee_euro":        0.0,
        "aporte_mensal":   50,
    },
    {
        "ticker":          "SGLN.L",
        "nome":            "Physical Gold ETC",
        "unidades":        7.1354790812,
        "euros_investidos":600.00,
        "moeda":           "GBP",
        "gbp_pence":       True,    # preço em GBp (pence) → ÷100
        "fee_euro":        0.0,
        "aporte_mensal":   50,
    },
    {
        "ticker":          "EMIM.AS",
        "nome":            "iShares Core MSCI EM IMI ETF",
        "unidades":        1.0669056535,
        "euros_investidos":50.0,
        "moeda":           "EUR",
        "gbp_pence":       False,
        "fee_euro":        0.0,
        "aporte_mensal":   50,
    },
    {
        "ticker":          "MEUD.PA",
        "nome":            "Core Stoxx Europe 600",
        "unidades":        0.1667500416,
        "euros_investidos":50.0,
        "moeda":           "EUR",
        "gbp_pence":       False,
        "fee_euro":        0.0,
        "aporte_mensal":   50,
    },
    {
        "ticker":          "SJPA.MI",
        "nome":            "iShares Core MSCI Japan IMI ETF",
        "unidades":        0.7322917215,
        "euros_investidos":50.0,
        "moeda":           "EUR",
        "gbp_pence":       False,
        "fee_euro":        0.0,
        "aporte_mensal":   50,
    },
]

HORIZONTE_ANOS        = [1, 3, 5, 10]
TAXA_CRESCIMENTO_BASE = {"pessimista": 0.03, "base": 0.08, "optimista": 0.15}
FORECAST_DAYS         = 3

print("✅ Carteira configurada (v5.0)")
print(f"   eToro:  {len(ETORO)} ativos")
print(f"   ETFs:   {len(ETF_ACUMULACAO)} ativos")
print(f"   Aporte mensal ETFs: €{sum(e['aporte_mensal'] for e in ETF_ACUMULACAO)}/mês")
print()
print("   Verificação rápida:")
for e in ETF_ACUMULACAO:
    flag = "GBp (÷100)" if e.get("gbp_pence") else e["moeda"]
    print(f"   {e['ticker']:<10} moeda: {flag}")
for a in ETORO:
    print(f"   {a['ticker']:<10} fee: €{a['fee_euro']}")

## BLOCO 4 — Caminhos + CSV + Pastas

Funciona tanto no **GitHub Actions** como localmente — sem Google Drive.

| Ficheiro | Conteúdo |
|----------|----------|
| `AnaliseV5/predictions_log.csv` | Histórico de previsões + resultados |
| `AnaliseV5/ensemble_weights.json` | Pesos por modelo e por horizonte (d1/d2/d3) |
| `AnaliseV5/AnaliseGraficos/` | Gráficos gerados por execução |


In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOCO 4 — Caminhos (GitHub Actions / Local)
#
# No GitHub Actions: /home/runner/work/<repo>/<repo>/AnaliseV5/
# Localmente:        ./AnaliseV5/
# ══════════════════════════════════════════════════════════════

GITHUB_ACTIONS = os.getenv("GITHUB_ACTIONS") == "true"
COLAB          = False   # Não usa Drive — ficheiros ficam no repositório

LOG_DIR      = Path("AnaliseV5")
GRAFICOS_DIR = LOG_DIR / "AnaliseGraficos"

LOG_DIR.mkdir(parents=True, exist_ok=True)
GRAFICOS_DIR.mkdir(parents=True, exist_ok=True)

PRED_LOG  = LOG_DIR / "predictions_log.csv"
WEIGHTS_F = LOG_DIR / "ensemble_weights.json"

PRED_COLS = [
    "ticker", "pred_date", "target_date", "horizon",
    "direction", "pred_price", "confidence",
    "actual_price", "correct",
    "model_rf", "model_gb", "model_lr"
]

if not PRED_LOG.exists():
    pd.DataFrame(columns=PRED_COLS).to_csv(PRED_LOG, index=False)
    print(f"✅ CSV criado: {PRED_LOG}")
else:
    n = len(pd.read_csv(PRED_LOG))
    print(f"✅ CSV carregado: {n} registos existentes")

DEFAULT_WEIGHTS = {
    "d1": {"rf": 1.0, "gb": 1.0, "lr": 1.0},
    "d2": {"rf": 1.0, "gb": 1.0, "lr": 1.0},
    "d3": {"rf": 1.0, "gb": 1.0, "lr": 1.0},
}

if WEIGHTS_F.exists():
    with open(WEIGHTS_F) as f:
        ensemble_weights = json.load(f)
    if "d1" not in ensemble_weights:
        ensemble_weights = DEFAULT_WEIGHTS.copy()
    print(f"✅ Pesos carregados:")
    for dk, dw in ensemble_weights.items():
        print(f"   {dk}: RF={dw['rf']:.2f}  GB={dw['gb']:.2f}  LR={dw['lr']:.2f}")
else:
    ensemble_weights = DEFAULT_WEIGHTS.copy()
    with open(WEIGHTS_F, 'w') as f:
        json.dump(ensemble_weights, f, indent=2)
    print(f"✅ Pesos inicializados (iguais por horizonte)")

env = "GitHub Actions" if GITHUB_ACTIONS else "Local"
print(f"\n✅ Ambiente: {env}")
print(f"   Dados:    {LOG_DIR.resolve()}")
print(f"   Gráficos: {GRAFICOS_DIR.resolve()}")


## BLOCO 5 — Download de preços + câmbio live + contexto de mercado

**Novidade v5:** descarrega VIX e SPY como contexto global.
Usa T-1 (dia anterior) para evitar data leakage — quando os mercados europeus abrem, só existe o fecho de NY do dia anterior.

In [ ]:
# ── Câmbio em tempo real ──────────────────────────────────────────────────
def get_fx_rate(pair_ticker, fallback):
    try:
        rate = yf.Ticker(pair_ticker).fast_info['last_price']
        if rate and rate > 0:
            return rate
    except:
        pass
    print(f"  ⚠️  {pair_ticker} indisponível — usando fallback {fallback}")
    return fallback

print("🔄 Taxas de câmbio...")
EUR_USD = get_fx_rate("EURUSD=X", 1.12)
EUR_GBP = get_fx_rate("EURGBP=X", 0.85)
GBP_EUR = 1 / EUR_GBP
print(f"   EUR/USD = {EUR_USD:.4f}")
print(f"   EUR/GBP = {EUR_GBP:.4f}  (1 GBP = {GBP_EUR:.4f} EUR)")

def to_eur(preco, moeda, gbp_pence=False):
    """Converte preço para EUR. gbp_pence=True se em pence (GBp)."""
    if moeda == "USD": return preco / EUR_USD
    if moeda == "GBP": return (preco / 100 * GBP_EUR) if gbp_pence else (preco * GBP_EUR)
    return preco

# ── Contexto global: VIX + SPY ────────────────────────────────────────────
print("\n🔄 Contexto de mercado (VIX, SPY)...")
context_data = {}

for ctx_ticker, ctx_name in [("^VIX", "vix"), ("SPY", "spy")]:
    try:
        df_ctx = yf.download(ctx_ticker, period="2y", interval="1d",
                             auto_adjust=True, progress=False)
        if df_ctx.empty:
            raise ValueError("sem dados")
        if isinstance(df_ctx.columns, pd.MultiIndex):
            df_ctx.columns = df_ctx.columns.get_level_values(0)
        df_ctx.index = pd.to_datetime(df_ctx.index).normalize()
        df_ctx = df_ctx.sort_index()
        context_data[ctx_name] = df_ctx['Close'].rename(ctx_name)
        print(f"  ✅ {ctx_ticker:<6} ({ctx_name}): {len(df_ctx)} dias | último: {df_ctx['Close'].iloc[-1]:.2f}")
    except Exception as e:
        print(f"  ⚠️  {ctx_ticker}: {e} — features de contexto usarão valores neutros")

# ── Ativos da carteira ────────────────────────────────────────────────────
all_tickers = list({a["ticker"] for a in ETORO + ETF_ACUMULACAO})
print(f"\n🔄 Download de {len(all_tickers)} ativos...")
raw_data = {}

for ticker in all_tickers:
    try:
        df = yf.download(ticker, period="2y", interval="1d",
                         auto_adjust=True, progress=False)
        if df.empty:
            print(f"  ⚠️  {ticker}: sem dados")
            continue
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df.index = pd.to_datetime(df.index).normalize()
        df = df.sort_index()
        raw_data[ticker] = df

        etf_info = next((e for e in ETF_ACUMULACAO if e["ticker"] == ticker), None)
        if etf_info and etf_info["moeda"] == "GBP":
            eur_p = to_eur(df["Close"].iloc[-1], "GBP", etf_info.get("gbp_pence", False))
            print(f"  ✅ {ticker:<10} {len(df)}d | raw={df['Close'].iloc[-1]:.2f} | EUR={eur_p:.4f}€")
        else:
            print(f"  ✅ {ticker:<10} {len(df)}d | último={df['Close'].iloc[-1]:.2f}")
    except Exception as e:
        print(f"  ❌ {ticker}: {e}")

print(f"\n✅ {len(raw_data)} ativos + contexto prontos")

## BLOCO 6 — Feature Engineering

| Indicador | Descrição |
|-----------|-----------|
| SMA20/50 | Médias móveis — tendência curta e média |
| RSI14 | Sobrecomprado (>70) / Sobrevendido (<30) |
| MACD | Momentum e cruzamentos |
| Bollinger | Volatilidade e posição relativa do preço |
| ATR14 | Magnitude de movimento esperado |
| **spy_ret_1d** | **Retorno do S&P500 (T-1) — contexto global** |
| **vix_level** | **Nível de medo do mercado (T-1)** |
| **vix_change** | **Variação do VIX (T-1)** |
| target_d1/d2/d3 | Targets independentes por horizonte |

In [ ]:
FEATURE_COLS = [
    'SMA20_dist', 'SMA50_dist', 'sma_cross',
    'RSI14', 'MACD', 'MACD_sig', 'MACD_hist',
    'BB_width', 'BB_pos', 'ATR14',
    'ret_1d', 'ret_5d', 'vol_10d',
    'spy_ret_1d', 'vix_level', 'vix_change',   # contexto de mercado
]

def build_features(df, context_data):
    d = df.copy()

    # ── Médias Móveis ─────────────────────────────────────────────────────
    d['SMA20']      = d['Close'].rolling(20).mean()
    d['SMA50']      = d['Close'].rolling(50).mean()
    d['SMA20_dist'] = (d['Close'] - d['SMA20']) / d['SMA20']
    d['SMA50_dist'] = (d['Close'] - d['SMA50']) / d['SMA50']
    d['sma_cross']  = (d['SMA20'] > d['SMA50']).astype(int)

    # ── RSI ───────────────────────────────────────────────────────────────
    delta      = d['Close'].diff()
    gain       = delta.clip(lower=0).rolling(14).mean()
    loss       = (-delta.clip(upper=0)).rolling(14).mean()
    rs         = gain / loss.replace(0, np.nan)
    d['RSI14'] = 100 - (100 / (1 + rs))

    # ── MACD ──────────────────────────────────────────────────────────────
    ema12          = d['Close'].ewm(span=12, adjust=False).mean()
    ema26          = d['Close'].ewm(span=26, adjust=False).mean()
    d['MACD']      = ema12 - ema26
    d['MACD_sig']  = d['MACD'].ewm(span=9, adjust=False).mean()
    d['MACD_hist'] = d['MACD'] - d['MACD_sig']

    # ── Bollinger ─────────────────────────────────────────────────────────
    sma20          = d['Close'].rolling(20).mean()
    std20          = d['Close'].rolling(20).std()
    d['BB_upper']  = sma20 + 2 * std20
    d['BB_lower']  = sma20 - 2 * std20
    d['BB_width']  = (d['BB_upper'] - d['BB_lower']) / sma20
    band_range     = (d['BB_upper'] - d['BB_lower']).replace(0, np.nan)
    d['BB_pos']    = (d['Close'] - d['BB_lower']) / band_range

    # ── ATR ───────────────────────────────────────────────────────────────
    hl         = d['High'] - d['Low']
    hc         = (d['High'] - d['Close'].shift()).abs()
    lc         = (d['Low']  - d['Close'].shift()).abs()
    tr         = pd.concat([hl, hc, lc], axis=1).max(axis=1)
    d['ATR14'] = tr.rolling(14).mean()

    # ── Retornos ──────────────────────────────────────────────────────────
    d['ret_1d']  = d['Close'].pct_change(1)
    d['ret_5d']  = d['Close'].pct_change(5)
    d['vol_10d'] = d['ret_1d'].rolling(10).std()

    # ── Contexto de mercado (T-1 para evitar data leakage) ───────────────
    if 'spy' in context_data:
        spy_ret           = context_data['spy'].pct_change(1).shift(1)
        d['spy_ret_1d']   = spy_ret.reindex(d.index, method='ffill').values
    else:
        d['spy_ret_1d']   = 0.0

    if 'vix' in context_data:
        vix               = context_data['vix'].shift(1)
        vix_chg           = context_data['vix'].pct_change(1).shift(1)
        d['vix_level']    = vix.reindex(d.index, method='ffill').values
        d['vix_change']   = vix_chg.reindex(d.index, method='ffill').values
    else:
        d['vix_level']    = 20.0
        d['vix_change']   = 0.0

    # ── Targets independentes por horizonte (novidade v5) ─────────────────
    d['target_d1'] = (d['Close'].shift(-1) > d['Close']).astype(int)
    d['target_d2'] = (d['Close'].shift(-2) > d['Close']).astype(int)
    d['target_d3'] = (d['Close'].shift(-3) > d['Close']).astype(int)

    d = d.dropna()
    return d

featured_data = {}
for ticker, df in raw_data.items():
    fd = build_features(df, context_data)
    if len(fd) > 60:
        featured_data[ticker] = fd
        print(f"✅ {ticker:<10} {len(fd)} dias com features completas")
    else:
        print(f"⚠️  {ticker}: dados insuficientes ({len(fd)} dias)")

print(f"\n✅ Features prontas para {len(featured_data)} ativos")
print(f"   Features utilizadas: {len(FEATURE_COLS)} ({len(FEATURE_COLS)-3} técnicas + 3 contexto)")

## BLOCO 7 — Modelos ML por Horizonte

**Novidade v5:** D+1, D+2 e D+3 têm ensembles totalmente independentes.
Cada um treina com o seu próprio target — elimina a extrapolação que existia no v4.

| Modelo | Vantagem |
|--------|----------|
| Random Forest | Robusto a overfitting |
| Gradient Boosting | Captura padrões não-lineares |
| Logistic Regression | Estável, âncora contra sobreajuste |

In [ ]:
def treinar_horizonte(df, target_col, weights):
    """
    Treina ensemble para um horizonte específico.
    Remove as últimas N linhas sem target válido (N = nº de dias do horizonte).
    """
    day      = int(target_col[-1])
    df_train = df.iloc[:-day].copy() if day > 0 else df.copy()

    X        = df_train[FEATURE_COLS].values
    y        = df_train[target_col].values
    scaler   = RobustScaler()
    X_scaled = scaler.fit_transform(X)

    tscv      = TimeSeriesSplit(n_splits=5)
    acuracias = {"rf": [], "gb": [], "lr": []}

    models = {
        "rf": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1),
        "gb": GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42),
        "lr": LogisticRegression(C=0.1, max_iter=1000, random_state=42),
    }

    for train_idx, val_idx in tscv.split(X_scaled):
        X_tr, X_val = X_scaled[train_idx], X_scaled[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        for name, m in models.items():
            m.fit(X_tr, y_tr)
            acuracias[name].append(accuracy_score(y_val, m.predict(X_val)))

    for m in models.values():
        m.fit(X_scaled, y)

    acc_media = {k: np.mean(v) for k, v in acuracias.items()}
    X_last    = scaler.transform(df[FEATURE_COLS].iloc[[-1]].values)
    probs     = {name: m.predict_proba(X_last)[0][1] for name, m in models.items()}
    preds     = {name: "up" if p > 0.5 else "down" for name, p in probs.items()}

    total_w    = sum(weights.values())
    prob_ens   = sum(probs[k] * weights[k] for k in weights) / total_w
    direction  = "up" if prob_ens > 0.5 else "down"
    confidence = max(prob_ens, 1 - prob_ens)

    return models, scaler, acc_media, prob_ens, confidence, direction, preds, probs

# ── Treinar todos os ativos ───────────────────────────────────────────────
resultados_ml = {}
print("🔄 Treinando modelos (D+1, D+2, D+3 independentes)...\n")

for ticker in featured_data:
    df = featured_data[ticker]
    try:
        horizons = {}
        for day in [1, 2, 3]:
            target_col = f'target_d{day}'
            day_key    = f'd{day}'
            w          = ensemble_weights.get(day_key, {"rf": 1.0, "gb": 1.0, "lr": 1.0})
            _, _, accs, prob, conf, direction, preds_ind, probs_ind =                 treinar_horizonte(df, target_col, w)
            horizons[day] = {
                "direction":  direction,
                "confidence": conf,
                "prob":       prob,
                "acc_media":  accs,
                "preds_ind":  preds_ind,
                "probs_ind":  probs_ind,
            }

        close_now  = df['Close'].iloc[-1]
        atr        = df['ATR14'].iloc[-1]
        preds_dict = {}
        for day in [1, 2, 3]:
            h     = horizons[day]
            mag   = atr * 0.5 * np.sqrt(day)
            price = (close_now + mag) if h['direction'] == "up" else (close_now - mag)
            preds_dict[day] = (h['direction'], price, h['confidence'])

        resultados_ml[ticker] = {
            "close_now":  close_now,
            "last_date":  df.index[-1],
            "direction":  horizons[1]['direction'],
            "confidence": horizons[1]['confidence'],
            "prob":       horizons[1]['prob'],
            "preds_dict": preds_dict,
            "horizons":   horizons,
            "df":         df,
            "atr":        atr,
        }

        d1, d2, d3 = horizons[1], horizons[2], horizons[3]
        print(f"  ✅ {ticker:<10} | "
              f"D+1: {d1['direction']:4s} {d1['confidence']:.0%} "
              f"(RF:{d1['acc_media']['rf']:.0%} GB:{d1['acc_media']['gb']:.0%} LR:{d1['acc_media']['lr']:.0%}) | "
              f"D+2: {d2['direction']:4s} {d2['confidence']:.0%} | "
              f"D+3: {d3['direction']:4s} {d3['confidence']:.0%}")
    except Exception as e:
        print(f"  ❌ {ticker}: {e}")

print(f"\n✅ Modelos treinados: {len(resultados_ml)} ativos")

## BLOCO 8 — Validar previsões antigas + guardar novas

A cada execução:
1. Valida previsões passadas (preenche `actual_price` + `correct`)
2. Atualiza pesos com **decay temporal** por horizonte
3. Guarda as novas previsões de hoje

**Correções v4.2 mantidas:** FIX 1 (tickers removidos), FIX 2 (acurácia real), FIX 3 (mercado aberto)

**Novidade v5 — FIX 4:** decay exponencial — previsões recentes pesam mais, separado por d1/d2/d3.

In [ ]:
df_log       = pd.read_csv(PRED_LOG)
hoje         = pd.Timestamp.now().normalize()
updated      = 0
skipped_open = 0

# ── FIX 1: Validar tickers removidos da carteira ──────────────────────────
tickers_ativos    = set(featured_data.keys())
tickers_pendentes = set(df_log.loc[df_log['actual_price'].isna(), 'ticker'].unique())
tickers_ausentes  = tickers_pendentes - tickers_ativos

if tickers_ausentes:
    print(f"ℹ️  Tickers fora da carteira com previsões pendentes: {tickers_ausentes}")
    for ticker in tickers_ausentes:
        try:
            df_extra = yf.download(ticker, period="10d", auto_adjust=True, progress=False)
            if not df_extra.empty:
                if isinstance(df_extra.columns, pd.MultiIndex):
                    df_extra.columns = df_extra.columns.get_level_values(0)
                df_extra.index = pd.to_datetime(df_extra.index).normalize()
                featured_data[ticker] = df_extra
                print(f"   ✅ {ticker}: dados obtidos")
            else:
                print(f"   ⚠️  {ticker}: sem dados (ticker inválido)")
        except Exception as e:
            print(f"   ⚠️  {ticker}: {e}")

# ── 8A: Validar previsões passadas ────────────────────────────────────────
for idx, row in df_log.iterrows():
    if pd.notna(row.get('actual_price')):
        continue

    target_date = pd.to_datetime(row['target_date'])
    if target_date > hoje:
        continue

    ticker = row['ticker']
    if ticker not in featured_data:
        continue

    df_tick = featured_data[ticker]

    if target_date == hoje:
        future_prices = df_tick[df_tick.index == hoje]['Close']
        if future_prices.empty:
            skipped_open += 1
            continue
    else:
        future_prices = df_tick[df_tick.index >= target_date]['Close']

    if future_prices.empty:
        continue

    actual  = float(future_prices.iloc[0])
    correct = (
        (row['direction'] == 'up'   and actual > row['pred_price'] * 0.995) or
        (row['direction'] == 'down' and actual < row['pred_price'] * 1.005)
    )
    df_log.at[idx, 'actual_price'] = actual
    df_log.at[idx, 'correct']      = correct
    updated += 1

if updated > 0:
    df_log.to_csv(PRED_LOG, index=False)
    print(f"✅ {updated} previsões validadas")
if skipped_open > 0:
    print(f"⏳ {skipped_open} previsões aguardam fecho de mercado — validadas amanhã")
if updated == 0 and skipped_open == 0:
    print("ℹ️  Nenhuma previsão nova para validar")

# ── 8B: Atualizar pesos com decay temporal por horizonte (FIX 4) ──────────
def calc_horizon(row):
    try:
        return (pd.to_datetime(row['target_date']) - pd.to_datetime(row['pred_date'])).days
    except:
        return None

validadas_log = df_log[df_log['correct'].notna()].copy()
if 'horizon' not in validadas_log.columns or validadas_log['horizon'].isna().all():
    validadas_log['horizon'] = validadas_log.apply(calc_horizon, axis=1)

weights_updated = False
for day_n in [1, 2, 3]:
    day_key     = f'd{day_n}'
    validadas_h = validadas_log[validadas_log['horizon'] == day_n].tail(30)

    if len(validadas_h) < 5:
        continue

    n     = len(validadas_h)
    decay = np.exp(0.1 * np.arange(n))   # mais recente = maior peso
    decay = decay / decay.sum()

    new_weights = {}
    for model_col, key in [('model_rf', 'rf'), ('model_gb', 'gb'), ('model_lr', 'lr')]:
        if model_col not in validadas_h.columns:
            new_weights[key] = ensemble_weights[day_key].get(key, 1.0)
            continue
        modelo_certo = (
            (validadas_h[model_col].notna()) &
            (
                ((validadas_h[model_col] == validadas_h['direction']) & (validadas_h['correct'] == True)) |
                ((validadas_h[model_col] != validadas_h['direction']) & (validadas_h['correct'] == False))
            )
        ).astype(float).values
        acc_weighted         = (modelo_certo * decay).sum()
        new_weights[key]     = max(0.1, acc_weighted)

    total                    = sum(new_weights.values())
    ensemble_weights[day_key]= {k: round(v * 3.0 / total, 4) for k, v in new_weights.items()}
    weights_updated          = True

if weights_updated:
    with open(WEIGHTS_F, 'w') as f:
        json.dump(ensemble_weights, f, indent=2)
    print(f"✅ Pesos atualizados (decay temporal):")
    for dk, dw in ensemble_weights.items():
        print(f"   {dk}: RF={dw['rf']:.2f}  GB={dw['gb']:.2f}  LR={dw['lr']:.2f}")
else:
    print(f"ℹ️  Histórico insuficiente por horizonte (mín. 5) — pesos mantêm-se")

# ── 8C: Guardar novas previsões de hoje ──────────────────────────────────
novas_previsoes = []
hoje_str        = hoje.strftime('%Y-%m-%d')

for ticker, res in resultados_ml.items():
    for day, (direction, pred_price, conf) in res['preds_dict'].items():
        target_date_str = (hoje + pd.Timedelta(days=day)).strftime('%Y-%m-%d')
        ja_existe = (
            (df_log['ticker']      == ticker) &
            (df_log['pred_date']   == hoje_str) &
            (df_log['target_date'] == target_date_str)
        )
        if not df_log.empty and ja_existe.any():
            continue
        h_data = res['horizons'][day]
        novas_previsoes.append({
            "ticker":       ticker,
            "pred_date":    hoje_str,
            "target_date":  target_date_str,
            "horizon":      day,
            "direction":    direction,
            "pred_price":   round(pred_price, 4),
            "confidence":   round(conf, 4),
            "actual_price": np.nan,
            "correct":      np.nan,
            "model_rf":     h_data['preds_ind'].get('rf', ''),
            "model_gb":     h_data['preds_ind'].get('gb', ''),
            "model_lr":     h_data['preds_ind'].get('lr', ''),
        })

if novas_previsoes:
    df_novas = pd.DataFrame(novas_previsoes)
    df_log   = pd.concat([df_log, df_novas], ignore_index=True)
    df_log.to_csv(PRED_LOG, index=False)
    print(f"✅ {len(novas_previsoes)} novas previsões guardadas")
else:
    print("ℹ️  Previsões de hoje já existem no CSV")

# ── Resumo de acurácia ────────────────────────────────────────────────────
validadas_total = df_log[df_log['correct'].notna()]
if len(validadas_total) > 0:
    acc_global = validadas_total['correct'].astype(float).mean()
    print(f"\n📊 Acurácia global: {acc_global:.1%} ({len(validadas_total)} validadas)")
    print("\n   Por ativo:")
    for t, grp in validadas_total.groupby('ticker'):
        acc_t = grp['correct'].astype(float).mean()
        n_t   = len(grp)
        bar   = "█" * int(acc_t * 10) + "░" * (10 - int(acc_t * 10))
        print(f"   {t:<10} {bar} {acc_t:.0%} ({n_t} validadas)")

## BLOCO 9 — Análise da Carteira Atual

In [ ]:
print("=" * 72)
print(f"{'CARTEIRA eTORO':^72}")
print("=" * 72)
print(f"{'Ativo':<10} {'Preço Atual':>12} {'G/P €':>10} {'G/P %':>8} {'Breakeven':>12} {'Alvo +15%':>12}")
print("-" * 72)

resumo_etoro = []

for ativo in ETORO:
    ticker = ativo['ticker']
    if ticker not in resultados_ml:
        print(f"  ⚠️  {ticker}: sem dados de mercado")
        continue

    close         = resultados_ml[ticker]['close_now']
    moeda         = ativo['moeda']
    fee           = ativo['fee_euro']
    uni           = ativo['unidades']
    pa            = ativo['preco_abertura']
    pa_eur        = pa / EUR_USD if moeda == "USD" else pa
    close_eur     = close / EUR_USD if moeda == "USD" else close
    investido_eur = pa_eur * uni
    atual_eur     = close_eur * uni
    gp_eur        = atual_eur - investido_eur
    gp_pct        = gp_eur / investido_eur * 100
    breakeven_eur = pa_eur + (fee / uni if uni > 0 else 0)
    alvo_15_eur   = pa_eur * 1.15 + (fee / uni if uni > 0 else 0)

    print(f"  {ticker:<8} {close_eur:>12.2f}€ {gp_eur:>+10.2f}€ {gp_pct:>+8.2f}% "
          f"{breakeven_eur:>12.2f}€ {alvo_15_eur:>12.2f}€")

    resumo_etoro.append({
        "ticker": ticker, "nome": ativo['nome'],
        "close_eur": close_eur, "investido_eur": investido_eur,
        "atual_eur": atual_eur, "gp_eur": gp_eur, "gp_pct": gp_pct,
        "breakeven_eur": breakeven_eur, "alvo_15_eur": alvo_15_eur,
        "fee": fee, "unidades": uni,
    })

total_inv   = sum(r['investido_eur'] for r in resumo_etoro)
total_atual = sum(r['atual_eur'] for r in resumo_etoro)
total_gp    = total_atual - total_inv
print("-" * 72)
print(f"  {'TOTAL eTORO':<8} {'':>12} {total_gp:>+10.2f}€ {total_gp/total_inv*100:>+8.2f}%")
print()

print("=" * 72)
print(f"{'ETFs DE ACUMULAÇÃO':^72}")
print("=" * 72)
print(f"{'Ativo':<12} {'Preço EUR':>10} {'Investido':>10} {'Atual':>10} {'G/P €':>10} {'G/P %':>8}")
print("-" * 72)

resumo_etfs = []

for etf in ETF_ACUMULACAO:
    ticker    = etf['ticker']
    if ticker not in resultados_ml:
        print(f"  ⚠️  {ticker}: sem dados de mercado")
        continue

    close_raw     = resultados_ml[ticker]['close_now']
    moeda         = etf['moeda']
    uni           = etf['unidades']
    close_eur     = to_eur(close_raw, "GBP", etf.get("gbp_pence", False)) if moeda == "GBP" else close_raw
    atual_eur     = close_eur * uni
    investido_eur = etf['euros_investidos']
    gp_eur        = atual_eur - investido_eur
    gp_pct        = gp_eur / investido_eur * 100

    print(f"  {ticker:<10} {close_eur:>10.4f}€ {investido_eur:>10.2f}€ "
          f"{atual_eur:>10.2f}€ {gp_eur:>+10.2f}€ {gp_pct:>+8.2f}%")

    resumo_etfs.append({
        "ticker": ticker, "nome": etf['nome'],
        "close_eur": close_eur, "investido_eur": investido_eur,
        "atual_eur": atual_eur, "gp_eur": gp_eur, "gp_pct": gp_pct,
        "unidades": uni, "aporte_mensal": etf['aporte_mensal'],
    })

total_etf_inv   = sum(r['investido_eur'] for r in resumo_etfs)
total_etf_atual = sum(r['atual_eur'] for r in resumo_etfs)
total_etf_gp    = total_etf_atual - total_etf_inv
print("-" * 72)
print(f"  {'TOTAL ETFs':<10} {'':>10} {total_etf_inv:>10.2f}€ "
      f"{total_etf_atual:>10.2f}€ {total_etf_gp:>+10.2f}€ {total_etf_gp/total_etf_inv*100:>+8.2f}%")
print()
print(f"  CARTEIRA TOTAL: investido {total_inv+total_etf_inv:.2f}€ "
      f"| atual {total_atual+total_etf_atual:.2f}€ "
      f"| G/P {total_gp+total_etf_gp:+.2f}€")

## BLOCO 10 — Sinais de Saída

In [ ]:
print("=" * 72)
print(f"{'SINAIS DE SAÍDA — eTORO':^72}")
print("=" * 72)

for r in resumo_etoro:
    ticker    = r['ticker']
    close     = r['close_eur']
    breakeven = r['breakeven_eur']
    alvo15    = r['alvo_15_eur']
    gp_pct    = r['gp_pct']

    if ticker not in resultados_ml:
        continue

    res       = resultados_ml[ticker]
    df_t      = res['df']
    rsi       = df_t['RSI14'].iloc[-1]
    sma20     = df_t['SMA20'].iloc[-1]
    sma50     = df_t['SMA50'].iloc[-1]
    vix_now   = df_t['vix_level'].iloc[-1] if 'vix_level' in df_t.columns else 20
    h1        = res['horizons'][1]['direction']
    h2        = res['horizons'][2]['direction']
    h3        = res['horizons'][3]['direction']
    conf      = res['horizons'][1]['confidence']

    consenso = "📉 BEARISH nos 3 horizontes" if h1==h2==h3=="down" else \
               "📈 BULLISH nos 3 horizontes" if h1==h2==h3=="up"   else \
               f"➡️  Misto (D+1:{h1} D+2:{h2} D+3:{h3})"

    print(f"\n  {'─'*60}")
    print(f"  {ticker} ({r['nome']})")
    print(f"  {'─'*60}")
    print(f"  Preço atual:   {close:.2f}€")
    print(f"  Breakeven:     {breakeven:.2f}€  "
          f"({'✅ atingido' if close >= breakeven else f'falta {(breakeven-close)/close*100:.1f}%'})")
    print(f"  Alvo +15%:     {alvo15:.2f}€  "
          f"({'✅ atingido' if close >= alvo15 else f'falta {(alvo15-close)/close*100:.1f}%'})")
    print(f"  RSI(14):       {rsi:.1f}  "
          f"({'⚠️  SOBRECOMPRADO' if rsi > 70 else '⚠️  SOBREVENDIDO' if rsi < 30 else '✅ normal'})")
    print(f"  Tendência:     {'SMA20>SMA50 ↑' if sma20>sma50 else 'SMA20<SMA50 ↓'}")
    print(f"  VIX (T-1):     {vix_now:.1f}  {'⚠️  Medo elevado' if vix_now > 30 else '✅ normal'}")
    print(f"  ML horizonte:  {consenso}")
    print()

    if close >= alvo15 and rsi > 65:
        print("  🟢 VENDER — alvo atingido + RSI elevado")
    elif close >= breakeven and h1 == h2 == "down" and conf > 0.60:
        print("  🟡 ATENÇÃO — acima do breakeven mas ML bearish em D+1 e D+2")
    elif gp_pct < -20 and h1 == h2 == h3 == "down":
        print("  🔴 ATENÇÃO — queda prolongada + consenso bearish nos 3 horizontes")
    else:
        print("  ℹ️  MANTER — critérios de saída não atingidos")

## BLOCO 11 — Projeção 1 / 3 / 5 / 10 Anos

In [ ]:
def valor_futuro_dca(valor_atual, aporte_mensal, taxa_anual, anos):
    taxa_mensal      = (1 + taxa_anual) ** (1/12) - 1
    meses            = anos * 12
    capital_crescido = valor_atual * (1 + taxa_anual) ** anos
    aportes_futuro   = aporte_mensal * (((1 + taxa_mensal) ** meses - 1) / taxa_mensal) if taxa_mensal > 0 else aporte_mensal * meses
    return capital_crescido + aportes_futuro

print("=" * 90)
print(f"{'PROJEÇÃO eTORO':^90}")
print("=" * 90)
print(f"{'Ativo':<18} {'Atual €':>8} {'1 ano':>12} {'3 anos':>12} {'5 anos':>12} {'10 anos':>12}  Método")
print("-" * 90)

for r in resumo_etoro:
    ticker = r['ticker']
    atual  = r['atual_eur']
    if ticker in resultados_ml:
        df_t      = resultados_ml[ticker]['df']
        n_dias    = (df_t.index[-1] - df_t.index[0]).days
        taxa_hist = (df_t['Close'].iloc[-1] / df_t['Close'].iloc[0]) ** (365 / n_dias) - 1
        taxa_hist = max(-0.3, min(taxa_hist, 0.5))
        metodo    = f"hist={taxa_hist:+.0%}"
    else:
        taxa_hist = TAXA_CRESCIMENTO_BASE['base']
        metodo    = "base=8%"
    row_vals = [f"{valor_futuro_dca(atual, 0, taxa_hist, a):.0f}€" for a in HORIZONTE_ANOS]
    print(f"  {r['nome'][:16]:<16} {atual:>8.0f}€ {row_vals[0]:>12} {row_vals[1]:>12} {row_vals[2]:>12} {row_vals[3]:>12}  {metodo}")

print()
print("=" * 90)
print(f"{'PROJEÇÃO ETFs (com aportes mensais)':^90}")
print("=" * 90)
print(f"{'Ativo':<20} {'Atual €':>8} {'1 ano':>12} {'3 anos':>12} {'5 anos':>12} {'10 anos':>12}  Cenário")
print("-" * 90)

for r in resumo_etfs:
    ticker = r['ticker']
    atual  = r['atual_eur']
    aporte = r['aporte_mensal']
    if ticker in resultados_ml:
        df_t      = resultados_ml[ticker]['df']
        n_dias    = (df_t.index[-1] - df_t.index[0]).days
        taxa_hist = (df_t['Close'].iloc[-1] / df_t['Close'].iloc[0]) ** (365/n_dias) - 1
        taxa_hist = max(-0.3, min(taxa_hist, 0.5))
    else:
        taxa_hist = TAXA_CRESCIMENTO_BASE['base']
    row_vals = [f"{valor_futuro_dca(atual, aporte, taxa_hist, a):.0f}€" for a in HORIZONTE_ANOS]
    label    = f"hist={taxa_hist:+.0%} +{aporte}€/m"
    print(f"  {r['nome'][:18]:<18} {atual:>8.0f}€ {row_vals[0]:>12} {row_vals[1]:>12} {row_vals[2]:>12} {row_vals[3]:>12}  {label}")

print()
total_etf   = sum(r['atual_eur'] for r in resumo_etfs)
total_aport = sum(r['aporte_mensal'] for r in resumo_etfs)
print(f"  Cenários para ETFs (total atual + €{total_aport}/mês):")
for cenario, taxa in TAXA_CRESCIMENTO_BASE.items():
    vals = [f"{valor_futuro_dca(total_etf, total_aport, taxa, a):.0f}€" for a in HORIZONTE_ANOS]
    print(f"  {cenario:<12} ({taxa:+.0%}/ano): {' → '.join(vals)}")

## BLOCO 12 — Simulação DCA Detalhada

In [ ]:
for r in resumo_etfs:
    if r['aporte_mensal'] == 0:
        continue
    ticker = r['ticker']
    atual  = r['atual_eur']
    aporte = r['aporte_mensal']
    if ticker in resultados_ml:
        df_t      = resultados_ml[ticker]['df']
        n_dias    = (df_t.index[-1] - df_t.index[0]).days
        taxa_hist = (df_t['Close'].iloc[-1] / df_t['Close'].iloc[0]) ** (365/n_dias) - 1
        taxa_hist = max(-0.3, min(taxa_hist, 0.5))
    else:
        taxa_hist = TAXA_CRESCIMENTO_BASE['base']

    print(f"\n  DCA — {r['nome']} ({ticker})")
    print(f"  Capital: {atual:.2f}€ | Aporte: {aporte}€/mês | Taxa hist: {taxa_hist:+.1%}/ano")
    print(f"  {'Horizonte':<12} {'Capital (€)':>14} {'Aportes (€)':>16} {'Rendimento (€)':>16}")
    print(f"  {'-'*60}")
    for anos in [1, 2, 3, 5, 10]:
        vf      = valor_futuro_dca(atual, aporte, taxa_hist, anos)
        aportes = aporte * anos * 12
        rend    = vf - atual - aportes
        print(f"  {anos} ano(s)      {vf:>14.2f}€ {aportes:>16.2f}€ {rend:>+16.2f}€")

## BLOCO 13 — Visualizações → AnaliseGraficos

In [ ]:
plt.rcParams.update({
    'figure.facecolor': '#0f0f0f', 'axes.facecolor': '#1a1a1a',
    'axes.edgecolor':   '#444',    'text.color':     'white',
    'xtick.color':      '#aaa',    'ytick.color':    '#aaa',
    'grid.color':       '#333',    'axes.labelcolor':'white',
})

def plot_ativo(ticker, res, df_log_full):
    df_plot   = res['df'].tail(120)
    close_now = res['close_now']
    last_date = res['last_date']
    ativo_info= next((a for a in ETORO + ETF_ACUMULACAO if a['ticker'] == ticker), None)
    nome      = ativo_info['nome'] if ativo_info else ticker

    fig = plt.figure(figsize=(14, 10), facecolor='#0f0f0f')
    gs  = gridspec.GridSpec(4, 1, height_ratios=[4, 1.5, 1.5, 1.5], hspace=0.08)
    fig.suptitle(f"{ticker} — {nome}", color='white', fontsize=13, fontweight='bold')

    ax1 = fig.add_subplot(gs[0])
    ax1.plot(df_plot.index, df_plot['Close'],  color='#00bfff', linewidth=1.5, label='Preço')
    ax1.plot(df_plot.index, df_plot['SMA20'],  color='#ffa500', linewidth=1,   label='SMA20', alpha=0.8)
    ax1.plot(df_plot.index, df_plot['SMA50'],  color='#ff6b6b', linewidth=1,   label='SMA50', alpha=0.8)
    ax1.fill_between(df_plot.index, df_plot['BB_lower'], df_plot['BB_upper'],
                     alpha=0.06, color='cyan', label='Bollinger')

    if ativo_info:
        pa_raw = ativo_info.get('preco_abertura', None)
        if pa_raw:
            ax1.axhline(pa_raw, color='#cc44ff', linestyle=':', linewidth=1.2,
                        alpha=0.8, label=f'Abertura ({pa_raw:.2f})')

    pred_x = [last_date]
    pred_y = [close_now]
    for day, (direction, pred_price, conf) in res['preds_dict'].items():
        fdate  = last_date + pd.Timedelta(days=day)
        color  = '#00ff88' if direction == 'up' else '#ff4444'
        marker = '^' if direction == 'up' else 'v'
        ax1.scatter(fdate, pred_price, color=color, s=120, zorder=6, marker=marker,
                    label=f'D+{day}:{direction}({conf:.0%})')
        pred_x.append(fdate)
        pred_y.append(pred_price)
    ax1.plot(pred_x, pred_y, color='gray', linestyle=':', linewidth=1, alpha=0.4)

    df_tick_log = df_log_full[df_log_full['ticker'] == ticker].copy()
    validated   = df_tick_log[df_tick_log['correct'].notna()]
    if not validated.empty:
        corr = validated[validated['correct'] == True]
        err  = validated[validated['correct'] == False]
        if not corr.empty:
            ax1.scatter(pd.to_datetime(corr['target_date']), corr['pred_price'],
                        marker='o', color='lime', s=50, alpha=0.6, label='Prev. correta')
        if not err.empty:
            ax1.scatter(pd.to_datetime(err['target_date']), err['pred_price'],
                        marker='x', color='red', s=50, alpha=0.6, label='Prev. errada')

    ax1.set_ylabel('Preço', color='white')
    ax1.legend(fontsize=7, loc='upper left', ncol=4,
               facecolor='#1a1a1a', labelcolor='white', edgecolor='#444')
    ax1.grid(True, alpha=0.2)

    ax2 = fig.add_subplot(gs[1], sharex=ax1)
    ax2.plot(df_plot.index, df_plot['RSI14'], color='#cc44ff', linewidth=1)
    ax2.axhline(70, color='red',   linestyle='--', linewidth=0.8, alpha=0.7)
    ax2.axhline(30, color='green', linestyle='--', linewidth=0.8, alpha=0.7)
    ax2.fill_between(df_plot.index, 70, df_plot['RSI14'].clip(lower=70), alpha=0.15, color='red')
    ax2.fill_between(df_plot.index, df_plot['RSI14'].clip(upper=30), 30, alpha=0.15, color='green')
    ax2.set_ylim(0, 100)
    ax2.set_ylabel('RSI', color='white')
    ax2.grid(True, alpha=0.2)

    ax3 = fig.add_subplot(gs[2], sharex=ax1)
    colors_hist = ['#00ff88' if v >= 0 else '#ff4444' for v in df_plot['MACD_hist']]
    ax3.bar(df_plot.index, df_plot['MACD_hist'], color=colors_hist, alpha=0.5, width=1)
    ax3.plot(df_plot.index, df_plot['MACD'],     color='#00bfff', linewidth=1, label='MACD')
    ax3.plot(df_plot.index, df_plot['MACD_sig'], color='orange',  linewidth=1, label='Signal')
    ax3.axhline(0, color='white', linewidth=0.5, alpha=0.5)
    ax3.set_ylabel('MACD', color='white')
    ax3.legend(fontsize=7, facecolor='#1a1a1a', labelcolor='white', edgecolor='#444')
    ax3.grid(True, alpha=0.2)

    ax4 = fig.add_subplot(gs[3])
    if len(validated) >= 3:
        val_s          = validated.sort_values('target_date').copy()
        val_s['cum_acc']= val_s['correct'].astype(float).expanding().mean() * 100
        ax4.plot(pd.to_datetime(val_s['target_date']), val_s['cum_acc'],
                 color='teal', linewidth=2, label='Acurácia acumulada')
        ax4.axhline(50, color='gray', linestyle='--', linewidth=0.8, alpha=0.7, label='Baseline 50%')
        ax4.set_ylim(0, 105)
        ax4.legend(fontsize=7, facecolor='#1a1a1a', labelcolor='white', edgecolor='#444')
        ax4.set_title('Acurácia das previsões', color='white', fontsize=9)
    else:
        ax4.text(0.5, 0.5, 'Acurácia disponível após primeiras previsões validadas',
                 ha='center', va='center', transform=ax4.transAxes, color='gray', fontsize=9)
    ax4.set_ylabel('Acurácia %', color='white')
    ax4.grid(True, alpha=0.2)

    plt.tight_layout()
    safe      = ticker.replace('.', '_').replace('-', '_')
    data_hoje = datetime.now().strftime("%Y%m%d")
    path      = GRAFICOS_DIR / f"{safe}_v5_{data_hoje}.png"
    plt.savefig(path, dpi=120, bbox_inches='tight', facecolor='#0f0f0f')
    plt.show()
    print(f"  📊 {ticker} → {path}")

df_log_loaded = pd.read_csv(PRED_LOG)
for ticker in resultados_ml:
    plot_ativo(ticker, resultados_ml[ticker], df_log_loaded)

## BLOCO 14 — Resumo Final das Previsões ML

In [ ]:
print("=" * 95)
print(f"{'PREVISÕES ML v5 — RESUMO FINAL':^95}")
print("=" * 95)
print(f"  {'Ativo':<10} {'Preço':>10}  {'D+1':>12} {'D+2':>12} {'D+3':>12}  {'Consenso':<14}  {'RF':>6} {'GB':>6} {'LR':>6}")
print("-" * 95)

for ticker, res in resultados_ml.items():
    close = res['close_now']
    preds = res['preds_dict']
    pind  = res['horizons'][1]['preds_ind']

    d = {}
    for day in [1, 2, 3]:
        direction, _, conf = preds[day]
        d[day] = f"{direction.upper()}({conf:.0%})"

    dirs     = [preds[day][0] for day in [1, 2, 3]]
    consenso = "📈 BULLISH" if all(x == "up" for x in dirs) else \
               "📉 BEARISH" if all(x == "down" for x in dirs) else "➡️  MISTO"

    rf = pind.get('rf', '?').upper()
    gb = pind.get('gb', '?').upper()
    lr = pind.get('lr', '?').upper()

    print(f"  {ticker:<10} {close:>10.2f}  {d[1]:>12} {d[2]:>12} {d[3]:>12}  {consenso:<14}  {rf:>6} {gb:>6} {lr:>6}")

print()
print(f"  Data: {datetime.now().strftime('%d/%m/%Y %H:%M')}")
for dk, dw in ensemble_weights.items():
    print(f"  Pesos {dk}: RF={dw['rf']:.2f}  GB={dw['gb']:.2f}  LR={dw['lr']:.2f}")
print()
print("  ⚠️  Previsões são probabilísticas. 55-60% já está acima do aleatório.")
print("  ⚠️  Nunca tomes decisões financeiras baseadas exclusivamente neste output.")